In [2]:
import re
import pandas as pd


STREET_WORDS = r"(?:street|st|avenue|ave|boulevard|blvd|road|rd|drive|dr|parkway|pkwy|place|pl)"
ADDRESS_RE = re.compile(rf"\b\d{{1,5}}(?:-\d{{1,5}})?\s+(?:[A-Za-z0-9]+\s+){{0,3}}{STREET_WORDS}\b", re.I)
STREET_RE = re.compile(rf"\b(?:[A-Za-z]+|\d{{1,3}}(?:st|nd|rd|th)?)\s+{STREET_WORDS}\b", re.I)
SKIP_FIRST = {"the", "a", "an", "my", "our", "this", "that", "every", "same", "main", "next"}
BOROUGHS = ["manhattan", "brooklyn", "queens", "bronx", "staten island"]
SEVERE = ["waist", "submerged", "stuck", "trapped", "emergency", "rescue", "cars", "river"]
TYPE_KW = [
    ("flood", r"flood|water|submerged|inundat|river|sewer|basement|drain"),
    ("power_outage", r"power|electric|outage|blackout|grid|transformer|substation"),
    ("road_blockage", r"road|bridge|highway|tree|branch|blocked|debris|lane"),
]
SOP = {   # PLACEHOLDER text. Real version = RAG over NYC protocol documents.
    "flood": "SOP-FLOOD-01 (placeholder): dispatch DEP crew, close affected road segment",
    "power_outage": "SOP-POWER-01 (placeholder): notify utility, flag dependent traffic signals",
    "road_blockage": "SOP-BLOCK-01 (placeholder): notify DOT, reroute emergency vehicles",
    "unspecified": "SOP-GENERAL-01 (placeholder): route to operator for review",
}

def extract_entities(text):
    location = None
    m = ADDRESS_RE.search(text)
    if m:
        location = m.group(0)
    else:
        for m in STREET_RE.finditer(text):
            if m.group(0).split()[0].lower() not in SKIP_FIRST:
                location = m.group(0)
                break
    borough = next((b.title() for b in BOROUGHS if b in text.lower()), None)
    severity = "high" if any(w in text.lower() for w in SEVERE) else "medium"
    return location, borough, severity

def keyword_type(text):
    for label, pat in TYPE_KW:
        if re.search(pat, text, flags=re.I):
            return label
    return "unspecified"

In [3]:
import pandas as pd
pd.set_option("display.max_colwidth", 140)

sf = pd.read_csv("data/processed/humaid_safety.csv")
haz = sf[sf["safety_label"] == "hazard"].sample(300, random_state=0).copy()

res = haz["text"].apply(extract_entities)
haz["location"] = [r[0] for r in res]
haz["borough"] = [r[1] for r in res]
haz["type"] = haz["text"].apply(keyword_type)

print("share with a street/address found:", round(haz["location"].notna().mean(), 2))
print(haz["type"].value_counts())
print("\n--- found a location (check each: right or wrong?) ---")
display(haz[haz["location"].notna()].head(15)[["text", "location"]])
print("\n--- no location found (is there one in the text?) ---")
display(haz[haz["location"].isna()].head(15)[["text"]])

share with a street/address found: 0.03
type
unspecified      210
flood             59
power_outage      20
road_blockage     11
Name: count, dtype: int64

--- found a location (check each: right or wrong?) ---


,text,location
16808,: Hurricane Irma: French St Martin 95% destroyed as Florida orders evacuations – live,French St
10342,"Hurricane Irma ripped roofs off, wrecked cars, destroyed the inside of houses and killed at least 4 in St. Martin",4 in St
11043,Kenny Chesneys St. John residence destroyed by Irma: Its just simply gone via,Chesneys St
18774,We are so happy to report that The Buccaneer has been spared of major damage after Hurricane Maria passed by St.,by St
14915,Hurricane Irma causes blackouts and flooding on St. Martin as winds destroy sturdiest buildings Live from,on St
19269,Aerial footage shows the damage caused to St. Croix in the US Virgin Islands by Hurricane Maria,to St
22210,: Good Caribbean news as St. Thomas Airport opens following hurricane damage. USVI Irma Maria travel,as St
14865,ICYMI: tornado damage along Indian River Drive in PalmBay WFTV Irma,River Drive
22476,Island of Puerto Rico destroyed by Hurricane Maria SHELTERINPLACE ALERT Theyre urging people to shelter in pl,in pl
19702,BREAKING: First pictures out of St. Croix and Vieques show the devastating Hurricane Maria damage,of St



--- no location found (is there one in the text?) ---


,text
7426,"Talk about bad luck - TWO homes ( Rockport, Houston) owned by ONE couple hit by HurricaneHarvey in THREE days. 1/2"
2706,Beira: Owners of destroyed restaurants and hotels request tax exemptions Mozambique Idai CycloneIdai ClicloneIdai
17232,I saw how little damage the port of Norfolk suffered in Maria &amp; hurricane has two Rs in it. I was an airlift crew chief. How bout u ...
8405,Flying over TX14 to assess the damage from Harvey. Its hard to put these sights into words
17760,": HURRICANE Maria destroyed Puerto Rico’s radar, a critical tool for forecasting GPWX"
23725,Haiti being damaged by hurricane after being destroyed by earthquake and the Clinton Foundation!
14070,"NEW: 2.7 million customers, or 27% of all power customers in Florida, currently without power, FL emergency management says. Irma"
6776,"Tips from for churches, ministries damaged in wake of Hurricane Harvey STORY:"
19302,": Hurricane Maria in Puerto Rico: Roofs and doors gone, 3.5M without power, heavy flooding"
17261,Puerto Rico cell phone companies struggle to restore service after HurricaneMaria damaged 90% cell sites


In [4]:
import spacy
nlp = spacy.load("en_core_web_sm")

def extract_places(text):
    return [e.text for e in nlp(text).ents if e.label_ in ("GPE", "LOC", "FAC")]

haz["places"] = haz["text"].apply(extract_places)
found = haz["places"].str.len() > 0
print("share with at least one place:", round(found.mean(), 2))

print("\n--- places found (check each: correct or wrong?) ---")
display(haz[found].head(20)[["text", "places"]])
print("\n--- nothing found (is there a place the model missed?) ---")
display(haz[~found].head(15)[["text"]])

share with at least one place: 0.59

--- places found (check each: correct or wrong?) ---


,text,places
7426,"Talk about bad luck - TWO homes ( Rockport, Houston) owned by ONE couple hit by HurricaneHarvey in THREE days. 1/2","[Rockport, Houston]"
17232,I saw how little damage the port of Norfolk suffered in Maria &amp; hurricane has two Rs in it. I was an airlift crew chief. How bout u ...,[Norfolk]
8405,Flying over TX14 to assess the damage from Harvey. Its hard to put these sights into words,[Harvey]
17760,": HURRICANE Maria destroyed Puerto Rico’s radar, a critical tool for forecasting GPWX",[Puerto Rico’s]
23725,Haiti being damaged by hurricane after being destroyed by earthquake and the Clinton Foundation!,[Haiti]
14070,"NEW: 2.7 million customers, or 27% of all power customers in Florida, currently without power, FL emergency management says. Irma",[Florida]
6776,"Tips from for churches, ministries damaged in wake of Hurricane Harvey STORY:",[Hurricane Harvey STORY]
19302,": Hurricane Maria in Puerto Rico: Roofs and doors gone, 3.5M without power, heavy flooding",[Puerto Rico]
17261,Puerto Rico cell phone companies struggle to restore service after HurricaneMaria damaged 90% cell sites,[Puerto Rico]
4036,The largest oil refinery in the US is shutting down as Hurricane Harvey causes more catastrophic flooding,[US]



--- nothing found (is there a place the model missed?) ---


,text
2706,Beira: Owners of destroyed restaurants and hotels request tax exemptions Mozambique Idai CycloneIdai ClicloneIdai
6072,A few tips from on how to save family treasures from water damage caused by HurricaneHarvey
15889,Sierra Leone: World Bank Technical Team presents damage and loss Assessment Report on August Mudslide and floods | a
10081,Heavy Rain Floods Privates Hospital in Kalaburgi: via
19587,Urgent‼️‼️ Follow and support thepausenetwork -- Hurricane Maria has caused catastrophic damage
18644,"NYC has 700,000 ppl of Puerto Rican descent1 in 12 city residents. City expects 1000s fleeing Maria damage. This is a climate impact."
177,"Farmers in some of hardest-hit areas of Mozambique receive maize&amp;bean seeds from , crucial for shorter second harvest to stave off f..."
12724,irma no LTE service at home. minor damage to my cottage but the yard is a mess. drove around the hood. many roads still blocked up.
2652,There is and its all under control. We pray no life be lost. The damage of property and crops we cant avoid. We pray for safety of the p...
4893,The damage from HurricaneHarvey is far from over yet one thing seems clear: investing in infrastructure is vital.


## Findings

- Regex street/address extractor on 300 HumAID hazard tweets found a location in only 3% of tweets, and none of the 10 matches was correct ("St" matched Saint in St. Martin; "pl" matched "place").
- spaCy NER (en_core_web_sm): at least one place tagged in 59% of the tweets. Hand-checked on the first 20: 22 of 27 tagged places correct (~80%); errors are storm names ("Harvey", "Hurricane Harvey STORY") and platform names ("YouTube").
- Of 15 tweets with no place found, 7 contained a real place that was missed (Beira, Mozambique, Sierra Leone, Kalaburgi, NYC, Barbuda, "P.R"). Small sample, judged by hand.
- Locations from text are mostly area-level (city, state, country), not street-level. NYC 311 reports have exact coordinates (97–99%).
- Next: filter storm/platform names, try a larger NER model or fine-tuned LLM, geocode places to coordinates, and extract streets/landmarks where present.